# Myeloid ONLY Ref-Query Merge Pipeline v1.4
**Changelog v1.4 (current):**
- `P0` FIXED: `celltypist_proba` now **index-aligned** to `adata_merged.obs_names` via `reindex()`, matching the scVI/scANVI latent pattern. Pure shape-check was silently wrong if CellTypist returned rows in a different order.
- `P1` FIXED: `BATCH_KEY` column existence checked **before** prefixing in Step 6. `ensure_batch_tissue()` runs post-concat so cannot rescue a missing column that would `KeyError` here first.
- `P1` FIXED: All label columns converted to `category` dtype **before** `write_h5ad()` to reduce file size and speed up groupby/plotting.

**Changelog v1.3:**
- `BUG-NEW-3 (P0)` FIXED: CellTypist promoted to first-class output branch — `celltypist_label_direct`, `celltypist_label_direct_filt`, `obsm["celltypist_proba"]`, `uns["celltypist_label_order"]`. Added `export_celltypist_direct_results()` helper.
- `BUG-NEW-4 (P1)` FIXED: `BATCH_KEY` values prefixed `"ref_"`/`"qry_"` before `sc.concat` to prevent sample-name collisions.
- Inherited all fixes from myeloid v1.2 (BUG-NEW-1/2 and NEW-BUG-A/B/C) and T cell v1.3 (BUG-1 through BUG-6)

## Cell 0 — Imports & Global Settings

In [1]:
import os
# BUG-NEW-2 FIX: os.environ thread variables MUST be set before any import torch.
os.environ["OMP_NUM_THREADS"]     = "1"
os.environ["MKL_NUM_THREADS"]     = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

import sys
import warnings
import json
import gc
import joblib
from pathlib import Path
from datetime import datetime
from typing import Optional, List, Dict, Any

import numpy as np
import pandas as pd
from scipy.sparse import issparse, csr_matrix
from scipy import sparse
from scipy.stats import entropy

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

import torch
import scanpy as sc
import scvi
import celltypist
from celltypist import models
from umap import UMAP

warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

print("=" * 80)
print("Myeloid ONLY Ref-Query Merge Pipeline v1.3")
print("=" * 80)

gpu_available = torch.cuda.is_available()
print(f"GPU available: {gpu_available}")
if gpu_available:
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

scvi.settings.dl_num_workers = 0
print(f"scVI dl_num_workers: {scvi.settings.dl_num_workers}")


/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/h2048/miniconda3/envs/scvi_env/lib/python3.10/site-packages/celltypist/classifier.py:11: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  from scanpy import __version__ as scv


Myeloid ONLY Ref-Query Merge Pipeline v1.3
GPU available: True
GPU Device: Tesla V100-SXM2-16GB
scVI dl_num_workers: 0


## Cell 1 — Configuration
> **Edit `REFERENCE_H5AD` and `QUERY_H5AD` before running.**

In [2]:
# ==============================================================================
# 1. CONFIGURATION
# ==============================================================================

# --- Input Files (EDIT THESE) ---
REFERENCE_H5AD = "/home/h2048/data/py/0128/myeloid_analysis_unified/results/subcluster_unified_v2_20260128/adata_myeloid_subclustered_FINAL_v2_20260128.h5ad"
QUERY_H5AD     = "/home/h2048/data/py/0127/scarches_mapping_FIXED_v1_2/subsets/myeloid_cells.h5ad"

# --- Column Configuration ---
REF_LABEL_COARSE = "cell_type_L2"
REF_LABEL_FINE   = "cell_type_L3"

BATCH_KEY  = "sample"
TISSUE_KEY = "tissue"

# --- Output Configuration ---
OUTPUT_DIR    = "/home/h2048/data/py/20260308/myeloid_only_merged_pipeline"
OUTPUT_PREFIX = "myeloid_only_merged"

# --- Cell Type Selection ---
INCLUDE_COARSE_TYPES = None  # e.g., ["Mono", "Macro"]

# --- Pipeline Parameters ---
N_HVG                = 4000
FORCE_MARKERS_IN_HVG = True

# scVI Parameters
SCVI_N_LATENT   = 100
SCVI_N_LAYERS   = 2
SCVI_N_HIDDEN   = 128
SCVI_DROPOUT    = 0.1
MAX_EPOCHS_SCVI = 400

# scANVI Parameters
MAX_EPOCHS_SCANVI  = 200
UNLABELED_CATEGORY = "Unknown"

# Training Parameters
BATCH_SIZE    = 256
LEARNING_RATE = 1e-3
WEIGHT_DECAY  = 0.0

# CellTypist Parameters
CELLTYPIST_MODEL         = "/home/h2048/data/source/reference/celltypist_models/Immune_All_Low.pkl"
CELLTYPIST_MAJORITY_VOTE = True

# --- CellTypist direct output branch (NEW in v1.3) ---
# run_celltypist_on_full_genes() now returns the predictions object; these keys
# control how export_celltypist_direct_results() writes them to adata_merged.
CELLTYPIST_DIRECT_LABEL_KEY = "celltypist_label_direct"
CELLTYPIST_DIRECT_FILT_KEY  = "celltypist_label_direct_filt"
CELLTYPIST_CONF_THRESHOLD   = 0.5     # cells below this -> "Unknown" in filt key
CELLTYPIST_SAVE_PROBA       = True    # saves obsm["celltypist_proba"]

# Query-only clustering resolution
QUERY_LEIDEN_RESOLUTION = 1.0

# --- Myeloid Score Thresholds ---
CLASSICAL_MONO_THRESHOLD    = 0.3
NONCLASSICAL_MONO_THRESHOLD = 0.3

# --- Myeloid Gene Signatures ---
MYELOID_CORE_MARKERS      = ["LYZ", "CD14", "CD33", "PTPRC", "ITGAM", "ITGAX"]
CLASSICAL_MONO_MARKERS    = ["CD14", "FCGR1A", "CCR2", "CD36", "SELL"]
# NEW-BUG-B FIX: "CD16" is protein name, not HGNC gene symbol (gene = FCGR3A).
NONCLASSICAL_MONO_MARKERS = ["FCGR3A", "FCER1G", "PICALM", "RHOC"]
INTERMEDIATE_MONO_MARKERS = ["CD14", "FCGR3A", "CD86", "HLA-DRA"]
MACROPHAGE_MARKERS        = ["CD68", "CD163", "MRC1", "MARCO", "MSR1", "FCGR2A", "APOE", "C1QA"]
M1_MACROPHAGE_MARKERS     = ["CD86", "CD80", "TNF", "IL1B", "NOS2", "CXCL9", "CXCL10"]
M2_MACROPHAGE_MARKERS     = ["CD163", "MRC1", "ARG1", "IL10", "TGFB1", "CCL22", "PPARG"]
CDC1_MARKERS              = ["CLEC9A", "XCR1", "WDFY4", "IRF8", "BATF3"]
CDC2_MARKERS              = ["CD1C", "FCER1A", "CLEC10A", "CD2", "ESAM"]
PDC_MARKERS               = ["LILRA4", "CLEC4C", "NRP1", "TCF4", "IRF7", "GZMB"]
NEUTROPHIL_MARKERS        = ["S100A8", "S100A9", "FCGR3B", "CSF3R", "CEACAM8", "CXCR2", "FCN1"]
MAST_CELL_MARKERS         = ["KIT", "TPSAB1", "TPSB2", "HPGDS", "MS4A2", "FCER1A", "CPA3"]
EOSINOPHIL_MARKERS        = ["EPX", "PRG2", "CLC", "CCL26", "SIGLEC8"]
PROLIF_MARKERS            = ["MKI67", "TOP2A", "PCNA"]

STRESS_SIGNATURE_GENES = [
    "HSPA1A", "HSPA1B", "HSPA8", "HSP90AA1", "HSP90AB1", "DNAJB1",
    "JUN", "JUNB", "JUND", "FOS", "FOSB", "EGR1", "IER2"
]

S_GENES = [
    "MCM5", "PCNA", "TYMS", "FEN1", "MCM2", "MCM4", "RRM1", "UHRF1",
    "GINS2", "MCM6", "CDCA7", "DTL", "PRIM1", "HELLS", "RFC2", "RPA2",
    "NASP", "RAD51AP1", "GMNN", "WDR76", "SLBP", "CCNE2", "UBR7",
    "POLD3", "MSH2", "ATAD2", "RAD51", "RRM2", "CDC45", "CDC6", "EXO1"
]

G2M_GENES = [
    "HMGB2", "CDK1", "NUSAP1", "UBE2C", "BIRC5", "TPX2", "TOP2A", "NDC80",
    "CKS2", "NUF2", "CKS1B", "MKI67", "TMPO", "CENPF", "TACC3", "FAM64A",
    "SMC4", "CCNB1", "CKAP2L", "CKAP2", "AURKB", "BUB1", "KIF11", "ANP32E"
]

FORCED_MARKERS = list(set(
    MYELOID_CORE_MARKERS +
    CLASSICAL_MONO_MARKERS + NONCLASSICAL_MONO_MARKERS + INTERMEDIATE_MONO_MARKERS +
    MACROPHAGE_MARKERS + M1_MACROPHAGE_MARKERS + M2_MACROPHAGE_MARKERS +
    CDC1_MARKERS + CDC2_MARKERS + PDC_MARKERS +
    NEUTROPHIL_MARKERS + MAST_CELL_MARKERS + EOSINOPHIL_MARKERS +
    PROLIF_MARKERS
))

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
sc.settings.seed   = RANDOM_SEED
scvi.settings.seed = RANDOM_SEED


Seed set to 42


## Cell 2 — Helper Functions

In [3]:
# ==============================================================================
# 2. HELPER FUNCTIONS
# ==============================================================================

def ensure_counts_layer(adata, counts_layer="counts"):
    # Robust counts validation with float32 tolerance.
    # BUG-6 FIX: sparse[:N] returns sub-sparse-matrix; must call .toarray().flatten().
    if counts_layer not in (adata.layers or {}):
        print(f"  WARNING: layers['{counts_layer}'] not found, checking .X...")
        if hasattr(adata, 'X') and adata.X is not None:
            if issparse(adata.X):
                X_sample = adata.X[:1000].toarray().flatten()
            else:
                X_sample = np.asarray(adata.X[:1000]).ravel()
            sample = np.asarray(X_sample, dtype=np.float64)

            if np.any(sample < 0):
                raise ValueError("adata.X contains negative values - not valid raw counts!")

            if np.allclose(sample, np.round(sample), atol=1e-6):
                print(f"  -> Auto-copying .X to layers['{counts_layer}']")
                if issparse(adata.X) and not isinstance(adata.X, csr_matrix):
                    adata.layers[counts_layer] = csr_matrix(adata.X)
                else:
                    adata.layers[counts_layer] = adata.X.copy()
            else:
                raise ValueError(
                    f"CRITICAL ERROR: layers['{counts_layer}'] not found and .X "
                    f"does not look like raw counts. "
                    f"Sample range: [{sample.min():.4f}, {sample.max():.4f}]"
                )
        else:
            raise ValueError(
                f"CRITICAL ERROR: layers['{counts_layer}'] not found and .X is None."
            )

    X_counts = adata.layers[counts_layer]
    if issparse(X_counts):
        sample_data = X_counts.data[:1000]
    else:
        sample_data = X_counts.flat[:1000]

    sample = np.asarray(sample_data, dtype=np.float64)

    # Defensive fix: all-zero sparse matrix produces empty sample_data
    if sample.size == 0:
        print("  WARNING: sampled counts data is empty; skipping integer-range check")
        if issparse(X_counts) and not isinstance(X_counts, csr_matrix):
            adata.layers[counts_layer] = csr_matrix(X_counts)
        return counts_layer

    if np.any(sample < 0):
        raise ValueError("counts contains negative values!")

    if not np.allclose(sample, np.round(sample), atol=1e-6):
        raise ValueError(
            "counts looks non-integer (possible normalized/log data). "
            f"Sample range: [{sample.min():.4f}, {sample.max():.4f}]"
        )

    if issparse(X_counts) and not isinstance(X_counts, csr_matrix):
        adata.layers[counts_layer] = csr_matrix(X_counts)

    return counts_layer


def ensure_batch_tissue(adata, batch_key, tissue_key):
    # Ensure batch and tissue columns exist, are NA-free, and are categorical.
    # BUG-7 FIX: added NA fill for batch_key; scVI errors on NA in categorical covariates.
    if batch_key not in adata.obs.columns:
        print(f"  WARNING: {batch_key} not found, creating placeholder")
        adata.obs[batch_key] = "unknown_batch"
    adata.obs[batch_key] = adata.obs[batch_key].astype("string").fillna("unknown_batch")
    adata.obs[batch_key] = adata.obs[batch_key].astype("category")
    if "unknown_batch" not in adata.obs[batch_key].cat.categories:
        adata.obs[batch_key] = adata.obs[batch_key].cat.add_categories(["unknown_batch"])

    if tissue_key not in adata.obs.columns:
        print(f"  WARNING: {tissue_key} not found, creating placeholder")
        adata.obs[tissue_key] = "unknown_tissue"
    adata.obs[tissue_key] = adata.obs[tissue_key].astype("string").fillna("unknown_tissue")
    adata.obs[tissue_key] = adata.obs[tissue_key].astype("category")
    if "unknown_tissue" not in adata.obs[tissue_key].cat.categories:
        adata.obs[tissue_key] = adata.obs[tissue_key].cat.add_categories(["unknown_tissue"])


def compute_module_score_efficient(adata, gene_list, score_name, counts_layer="counts"):
    # Memory-efficient module score via direct mean-expression.
    # BUG-1 FIX: sc.tl.score_genes needs background genes absent in marker-only subset.
    genes = [g for g in gene_list if g in adata.var_names]
    if len(genes) < 3:
        adata.obs[score_name] = 0.0
        adata.obs[f"{score_name}_norm"] = 0.0
        return

    gene_idx = adata.var_names.get_indexer(genes)
    gene_idx = gene_idx[gene_idx >= 0]

    if len(gene_idx) < 3:
        adata.obs[score_name] = 0.0
        adata.obs[f"{score_name}_norm"] = 0.0
        return

    X_subset   = adata.layers[counts_layer][:, gene_idx]
    var_subset = adata.var.iloc[gene_idx].copy()

    adata_tmp = sc.AnnData(X=X_subset.copy(), var=var_subset)
    sc.pp.normalize_total(adata_tmp, target_sum=1e4)
    sc.pp.log1p(adata_tmp)

    X_norm = adata_tmp.X
    if issparse(X_norm):
        mean_expr = np.asarray(X_norm.mean(axis=1)).flatten()
    else:
        mean_expr = np.asarray(X_norm).mean(axis=1).flatten()

    adata.obs[score_name] = mean_expr

    scores   = adata.obs[score_name]
    mn, mx   = float(scores.min()), float(scores.max())
    if mx > mn:
        adata.obs[f"{score_name}_norm"] = (scores - mn) / (mx - mn)
    else:
        adata.obs[f"{score_name}_norm"] = 0.0

    del adata_tmp, X_subset
    gc.collect()


def compute_monocyte_subtype_scores(adata):
    # Compute monocyte subtype scores.
    # NEW-BUG-C FIX: vectorized np.select replaces Python for-loop.
    print("  -> Computing monocyte subtype scores...")

    compute_module_score_efficient(adata, CLASSICAL_MONO_MARKERS,    "Classical_Mono_score")
    compute_module_score_efficient(adata, NONCLASSICAL_MONO_MARKERS, "NonClassical_Mono_score")
    compute_module_score_efficient(adata, INTERMEDIATE_MONO_MARKERS, "Intermediate_Mono_score")

    classical_high    = adata.obs["Classical_Mono_score_norm"]    > CLASSICAL_MONO_THRESHOLD
    nonclassical_high = adata.obs["NonClassical_Mono_score_norm"] > NONCLASSICAL_MONO_THRESHOLD
    intermediate_high = adata.obs["Intermediate_Mono_score_norm"] > 0.3

    conditions = [
        classical_high    & ~nonclassical_high,
        nonclassical_high & ~classical_high,
        classical_high    & nonclassical_high &  intermediate_high,
        classical_high    & nonclassical_high & ~intermediate_high,
    ]
    choices = ["Classical", "NonClassical", "Intermediate", "DoublePositive"]
    adata.obs["mono_subtype_by_score"] = np.select(conditions, choices, default="Negative")

    counts = adata.obs["mono_subtype_by_score"].value_counts()
    for label in ["Classical", "NonClassical", "Intermediate", "DoublePositive", "Negative"]:
        print(f"    {label}: {counts.get(label, 0)}")


def prepare_covariates(adata):
    # Prepare covariates for scVI training.
    print("  -> Validating counts...")
    ensure_counts_layer(adata, "counts")

    print("  -> Batch/Tissue...")
    ensure_batch_tissue(adata, BATCH_KEY, TISSUE_KEY)

    print("  -> Computing signature scores...")
    if "pct_counts_mt" not in adata.obs.columns:
        adata.var["mt"] = adata.var_names.str.startswith("MT-")
        sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, layer="counts")

    compute_monocyte_subtype_scores(adata)
    compute_module_score_efficient(adata, MACROPHAGE_MARKERS,     "Macrophage_score")
    compute_module_score_efficient(adata, NEUTROPHIL_MARKERS,     "Neutrophil_score")
    compute_module_score_efficient(adata, STRESS_SIGNATURE_GENES, "stress_score")

    if not all(k in adata.obs.columns for k in ["S_score", "G2M_score"]):
        s_in = [g for g in S_GENES   if g in adata.var_names]
        g_in = [g for g in G2M_GENES if g in adata.var_names]

        if len(s_in) >= 5 and len(g_in) >= 5:
            cc_union = list(dict.fromkeys(s_in + g_in))
            cc_idx   = adata.var_names.get_indexer(cc_union)
            X_cc     = adata.layers["counts"][:, cc_idx].copy()
            var_cc   = adata.var.iloc[cc_idx].copy()

            ad_tmp = sc.AnnData(X=X_cc, var=var_cc)
            sc.pp.normalize_total(ad_tmp, target_sum=1e4)
            sc.pp.log1p(ad_tmp)
            sc.tl.score_genes_cell_cycle(ad_tmp, s_genes=s_in, g2m_genes=g_in)

            adata.obs["S_score"]   = ad_tmp.obs["S_score"].values
            adata.obs["G2M_score"] = ad_tmp.obs["G2M_score"].values
            adata.obs["phase"]     = ad_tmp.obs["phase"].values

            del ad_tmp
            gc.collect()
        else:
            adata.obs["S_score"]   = 0.0
            adata.obs["G2M_score"] = 0.0
            adata.obs["phase"]     = "G1"


def run_celltypist_on_full_genes(adata_merged, model_name=CELLTYPIST_MODEL,
                                  majority_vote=True):
    # Run CellTypist on full gene matrix.
    # BUG-NEW-3 FIX: now returns the predictions object so the caller can pass it
    #   to export_celltypist_direct_results() for first-class output.
    # BUG-2 FIX: local-file-check before download_models avoids HTTP error on full path.
    print("\n[CellTypist] Starting annotation on FULL gene matrix...")

    if os.path.isfile(model_name):
        print(f"  -> Loading model from local path: {model_name}")
        model = models.Model.load(model_name)
    else:
        model_basename = os.path.basename(model_name)
        print(f"  -> Local model not found, downloading: {model_basename}")
        models.download_models(model=model_basename)
        model = models.Model.load(model_basename)

    model_genes = set(model.features)

    if "symbol_base" in adata_merged.var.columns:
        base_to_var = {}
        for vn, sb in zip(adata_merged.var_names, adata_merged.var["symbol_base"]):
            if sb in model_genes and sb not in base_to_var:
                base_to_var[sb] = vn
        available_genes = list(base_to_var.values())
        print(f"  -> {len(available_genes)}/{len(model_genes)} model genes matched via symbol_base "
              f"(direct var_names: {sum(g in model_genes for g in adata_merged.var_names)})")
    else:
        available_genes = [g for g in adata_merged.var_names if g in model_genes]
        print(f"  -> {len(available_genes)}/{len(model_genes)} model genes matched "
              f"(symbol_base column missing, using direct var_names)")

    if len(available_genes) < 100:
        raise ValueError(f"Too few overlapping genes ({len(available_genes)}) for CellTypist!")

    adata_ct = adata_merged[:, available_genes].copy()
    if "symbol_base" in adata_ct.var.columns:
        adata_ct.var_names = pd.Index(adata_ct.var["symbol_base"].values)
        adata_ct.var_names_make_unique()
    sc.pp.normalize_total(adata_ct, target_sum=1e4)
    sc.pp.log1p(adata_ct)

    predictions = celltypist.annotate(
        adata_ct, model=model, majority_voting=majority_vote, mode='best match'
    )

    pl = predictions.predicted_labels

    if isinstance(pl, pd.DataFrame):
        pred_labels = (
            pl["predicted_labels"].astype(str).values
            if "predicted_labels" in pl.columns
            else pl.iloc[:, 0].astype(str).values
        )
        conf_values     = predictions.probability_matrix.max(axis=1).values
        majority_labels = (
            pl["majority_voting"].astype(str).values
            if majority_vote and "majority_voting" in pl.columns
            else None
        )
    else:
        pred_labels     = pl.astype(str).values
        conf_values     = predictions.probability_matrix.max(axis=1).values
        majority_labels = None

    adata_merged.obs["celltypist_pred"]       = pred_labels
    adata_merged.obs["celltypist_confidence"] = conf_values
    if majority_labels is not None:
        adata_merged.obs["celltypist_majority"] = majority_labels

    print("  -> CellTypist predictions added")
    print(pd.Series(pred_labels).value_counts().head(10))

    del adata_ct
    gc.collect()
    return predictions


def export_celltypist_direct_results(
    adata_merged,
    predictions,
    label_key=CELLTYPIST_DIRECT_LABEL_KEY,
    filt_key=CELLTYPIST_DIRECT_FILT_KEY,
    prefer_majority=True,
    conf_threshold=CELLTYPIST_CONF_THRESHOLD,
    save_proba=CELLTYPIST_SAVE_PROBA,
):
    # BUG-NEW-3 FIX: promote CellTypist to first-class output branch.
    # P0 FIX (v1.4): probability matrix explicitly index-aligned to adata_merged.obs_names,
    #   matching the scVI/scANVI reindex pattern.  Shape-only check was silently wrong if
    #   CellTypist returned rows in a different order.
    print("  -> Exporting CellTypist direct output branch...")

    # Source label column — build as an indexed Series for safe alignment
    if prefer_majority and "celltypist_majority" in adata_merged.obs.columns:
        raw_labels = pd.Series(
            adata_merged.obs["celltypist_majority"].astype(str).values,
            index=adata_merged.obs_names,
            dtype="object"
        )
        source = "celltypist_majority"
    else:
        raw_labels = pd.Series(
            adata_merged.obs["celltypist_pred"].astype(str).values,
            index=adata_merged.obs_names,
            dtype="object"
        )
        source = "celltypist_pred"

    adata_merged.obs[label_key] = raw_labels.astype("category")

    # Filtered version: low-confidence cells -> "Unknown"
    conf = pd.Series(
        adata_merged.obs["celltypist_confidence"].values,
        index=adata_merged.obs_names,
        dtype="float32"
    )
    filt_labels = raw_labels.where(conf >= conf_threshold, "Unknown")
    adata_merged.obs[filt_key] = filt_labels.astype("category")

    n_low = int((conf < conf_threshold).sum())
    print(f"    {label_key}: {adata_merged.obs[label_key].nunique()} unique types")
    print(f"    {filt_key}: {n_low} cells below conf threshold {conf_threshold} -> 'Unknown'")

    # Probability matrix — build as DataFrame and explicitly align to obs_names
    proba_mat = predictions.probability_matrix
    if isinstance(proba_mat, pd.DataFrame):
        proba_df = proba_mat.copy()
    else:
        proba_df = pd.DataFrame(
            np.asarray(proba_mat, dtype=np.float32),
            index=adata_merged.obs_names,
            columns=[f"ct_{i}" for i in range(np.asarray(proba_mat).shape[1])]
        )

    if len(proba_df) != adata_merged.n_obs:
        raise ValueError(
            f"CellTypist probability matrix row count mismatch: "
            f"{len(proba_df)} vs {adata_merged.n_obs}"
        )

    # Explicit index alignment — same pattern as scVI/scANVI latent reindex
    if not proba_df.index.equals(adata_merged.obs_names):
        try:
            proba_df = proba_df.reindex(adata_merged.obs_names)
        except Exception:
            proba_df.index = adata_merged.obs_names  # fallback: assume same order

    if proba_df.isna().any().any():
        raise ValueError("CellTypist probability matrix contains NaN after alignment!")

    label_order = [str(x) for x in proba_df.columns]

    if save_proba:
        adata_merged.obsm["celltypist_proba"] = proba_df.values.astype(np.float32)
        print(f"    celltypist_proba saved: shape {proba_df.shape}")

    adata_merged.uns["celltypist_label_order"] = label_order
    adata_merged.uns["celltypist_direct"] = {
        "label_key":        label_key,
        "filtered_key":     filt_key,
        "source_column":    source,
        "conf_threshold":   conf_threshold,
        "n_types":          len(label_order),
        "n_low_conf_cells": n_low,
    }

    return source


def compute_novelty_scores(adata_merged, proba_df):
    # Compute entropy-based novelty scores for query cells.
    # Defensive check: ensure no non-finite values propagated from upstream alignment.
    print("  -> Computing novelty scores (entropy-based)...")

    if not np.isfinite(proba_df.values).all():
        raise ValueError("Non-finite values found in probability matrix for novelty scoring")

    proba_array = proba_df.values
    epsilon     = 1e-10
    entropies   = entropy(proba_array + epsilon, axis=1)

    adata_merged.obs["scanvi_entropy"] = entropies

    ent_min, ent_max = entropies.min(), entropies.max()
    if ent_max > ent_min:
        adata_merged.obs["novelty_score"] = (entropies - ent_min) / (ent_max - ent_min)
    else:
        adata_merged.obs["novelty_score"] = 0.0

    query_mask   = adata_merged.obs["data_source"] == "query"
    high_novelty = (adata_merged.obs["novelty_score"] > 0.7) & query_mask
    adata_merged.obs["is_potentially_novel"] = high_novelty

    print(f"    High novelty query cells: {high_novelty.sum()}")


def run_query_only_leiden(adata_merged, resolution=QUERY_LEIDEN_RESOLUTION):
    # Run Leiden on query cells only via scVI latent space.
    # BUG-4 FIX: query_mask.values (numpy bool array) for safe numpy obsm indexing.
    print(f"\n[Novelty Detection] Running query-only Leiden (resolution={resolution})...")

    query_mask  = adata_merged.obs["data_source"] == "query"
    query_cells = adata_merged.obs_names[query_mask]

    if len(query_cells) < 10:
        print("  -> Too few query cells, skipping query-only clustering")
        adata_merged.obs["leiden_query"] = "N/A"
        return

    X_scVI_query = adata_merged.obsm["X_scVI"][query_mask.values]

    adata_qry_tmp = sc.AnnData(
        X=X_scVI_query,
        obs=adata_merged.obs.loc[query_cells].copy()
    )
    adata_qry_tmp.obsm["X_scVI"] = X_scVI_query

    sc.pp.neighbors(adata_qry_tmp, use_rep="X_scVI", n_neighbors=30,
                    random_state=RANDOM_SEED)
    sc.tl.leiden(adata_qry_tmp, resolution=resolution, random_state=RANDOM_SEED)

    leiden_full = pd.Series("N/A", index=adata_merged.obs_names, dtype="object")
    leiden_full.loc[query_cells] = "qry_" + adata_qry_tmp.obs["leiden"].astype(str)
    adata_merged.obs["leiden_query"] = leiden_full.values

    print(f"  -> Found {adata_qry_tmp.obs['leiden'].nunique()} query-only clusters")
    print(adata_merged.obs["leiden_query"].value_counts().head(10))

    del adata_qry_tmp
    gc.collect()


## Cell 3 — Initialization

In [4]:
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)
print(f"Output directory: {output_dir}")


Output directory: /home/h2048/data/py/20260308/myeloid_only_merged_pipeline


## Cell 4 — Step 1: Load Reference and Query

In [5]:
print("\n[Step 1] Loading Reference and Query...")

print(f"  Loading reference: {REFERENCE_H5AD}")
adata_ref = sc.read_h5ad(REFERENCE_H5AD)
adata_ref.var_names_make_unique()
print(f"    Reference shape: {adata_ref.shape}")

print(f"  Loading query: {QUERY_H5AD}")
adata_qry = sc.read_h5ad(QUERY_H5AD)
adata_qry.var_names_make_unique()
print(f"    Query shape: {adata_qry.shape}")



[Step 1] Loading Reference and Query...
  Loading reference: /home/h2048/data/py/0128/myeloid_analysis_unified/results/subcluster_unified_v2_20260128/adata_myeloid_subclustered_FINAL_v2_20260128.h5ad
    Reference shape: (54743, 35112)
  Loading query: /home/h2048/data/py/0127/scarches_mapping_FIXED_v1_2/subsets/myeloid_cells.h5ad
    Query shape: (111272, 83690)


## Cell 5 — Step 2: Subset Reference (if needed)

In [6]:
if INCLUDE_COARSE_TYPES is not None and REF_LABEL_COARSE in adata_ref.obs.columns:
    print(f"\n[Step 2] Subsetting reference to: {INCLUDE_COARSE_TYPES}")
    mask      = adata_ref.obs[REF_LABEL_COARSE].isin(INCLUDE_COARSE_TYPES)
    adata_ref = adata_ref[mask].copy()
    print(f"    Reference after subset: {adata_ref.shape}")
else:
    print("\n[Step 2] No subsetting applied.")



[Step 2] No subsetting applied.


## Cell 6 — Step 3: Prepare Labels

In [7]:
print("\n[Step 3] Preparing labels...")

if REF_LABEL_COARSE in adata_ref.obs.columns:
    adata_ref.obs["cell_type_coarse"] = adata_ref.obs[REF_LABEL_COARSE].astype(str)
    print(f"  -> Coarse labels from: {REF_LABEL_COARSE}")
    print(adata_ref.obs["cell_type_coarse"].value_counts())
else:
    print(f"  WARNING: {REF_LABEL_COARSE} not found, using 'Myeloid'")
    adata_ref.obs["cell_type_coarse"] = "Myeloid"

if REF_LABEL_FINE in adata_ref.obs.columns:
    adata_ref.obs["cell_type_fine"] = adata_ref.obs[REF_LABEL_FINE].astype(str)
    print(f"  -> Fine labels from: {REF_LABEL_FINE}")
else:
    adata_ref.obs["cell_type_fine"] = adata_ref.obs["cell_type_coarse"]
    print("  -> Using coarse labels as fine labels")

adata_qry.obs["cell_type_coarse"] = UNLABELED_CATEGORY
adata_qry.obs["cell_type_fine"]   = UNLABELED_CATEGORY



[Step 3] Preparing labels...
  -> Coarse labels from: cell_type_L2
cell_type_coarse
Alveolar macrophages       24146
Macrophages                13895
Classical monocytes         8909
Mast cells                  3730
Non-classical monocytes     2029
DC2                         1033
Intestinal macrophages       386
pDC                          336
DC                           279
Name: count, dtype: int64
  -> Fine labels from: cell_type_L3


## Cell 7 — Step 4: Find Common Genes (deterministic order)

In [8]:
print("\n[Step 4] Finding common genes...")

qry_set      = set(adata_qry.var_names)
common_genes = [g for g in adata_ref.var_names if g in qry_set]

print(f"  Reference: {adata_ref.n_vars:,}, Query: {adata_qry.n_vars:,}, Common: {len(common_genes):,}")

if len(common_genes) < 1000:
    raise ValueError(f"Too few common genes ({len(common_genes)}). Check gene naming!")

adata_ref = adata_ref[:, common_genes].copy()
adata_qry = adata_qry[:, common_genes].copy()



[Step 4] Finding common genes...
  Reference: 35,112, Query: 83,690, Common: 33,559


## Cell 8 — Step 5: Validate Counts

In [9]:
print("\n[Step 5] Validating counts...")
ensure_counts_layer(adata_ref, "counts")
ensure_counts_layer(adata_qry, "counts")



[Step 5] Validating counts...
  -> Auto-copying .X to layers['counts']


'counts'

## Cell 9 — Step 6: Concatenate
> **[BUG-NEW-4 P1]** `BATCH_KEY` prefixed `"ref_"`/`"qry_"` before `sc.concat` (v1.3)
> **[P1 v1.4]** column existence checked before prefixing

In [10]:
print("\n[Step 6] Concatenating...")

# BUG-NEW-4 FIX: Prefix BATCH_KEY values with ref_/qry_ to prevent sample-name
# collisions across datasets (e.g. both may have sample="P1"). Without this prefix,
# scVI treats identically-named samples from ref and query as the same batch.
# P1 FIX (v1.4): check column existence BEFORE prefixing; ensure_batch_tissue() runs
# post-concat so cannot rescue a missing column that explodes here first.
for ad, prefix in [(adata_ref, "ref_"), (adata_qry, "qry_")]:
    if BATCH_KEY not in ad.obs.columns:
        print(f"  WARNING: {BATCH_KEY} missing before concat, creating placeholder")
        ad.obs[BATCH_KEY] = "unknown_batch"
    ad.obs[BATCH_KEY] = prefix + ad.obs[BATCH_KEY].astype("string").fillna("unknown_batch")

adata_ref.obs_names = pd.Index([f"ref_{x}" for x in adata_ref.obs_names])
adata_qry.obs_names = pd.Index([f"qry_{x}" for x in adata_qry.obs_names])

adata_merged = sc.concat(
    {"reference": adata_ref, "query": adata_qry},
    axis=0, join="inner", merge="unique", label="data_source"
)

print(f"  Merged: {adata_merged.shape}")
print(f"  Reference: {(adata_merged.obs['data_source'] == 'reference').sum():,}")
print(f"  Query:     {(adata_merged.obs['data_source'] == 'query').sum():,}")

del adata_ref, adata_qry
gc.collect()



[Step 6] Concatenating...
  Merged: (166015, 33559)
  Reference: 54,743
  Query:     111,272


47852

## Cell 10 — Step 7: Prepare Covariates

In [11]:
print("\n[Step 7] Preparing covariates...")
prepare_covariates(adata_merged)

print("  -> Adding symbol_base column...")
adata_merged.var["symbol_base"] = adata_merged.var_names.str.replace(r"-\d+$", "", regex=True)



[Step 7] Preparing covariates...
  -> Validating counts...
  -> Batch/Tissue...
  -> Computing signature scores...
  -> Computing monocyte subtype scores...
    Classical: 14053
    NonClassical: 50629
    Intermediate: 22786
    DoublePositive: 2696
    Negative: 75851
  -> Adding symbol_base column...


## Cell 11 — Step 8: HVG Selection

In [12]:
print("\n[Step 8] Selecting HVGs...")
hvg_method = "unknown"
try:
    sc.pp.highly_variable_genes(
        adata_merged, layer="counts", n_top_genes=N_HVG,
        batch_key=BATCH_KEY, flavor="seurat_v3", subset=False
    )
    hvg_method = "batch_seurat_v3"
except Exception as e1:
    print(f"  -> batch-aware failed ({str(e1)[:50]}), trying standard...")
    try:
        sc.pp.highly_variable_genes(
            adata_merged, layer="counts", n_top_genes=N_HVG,
            flavor="seurat_v3", subset=False
        )
        hvg_method = "standard_seurat_v3"
    except Exception as e2:
        print(f"  -> standard failed ({str(e2)[:50]}), fallback to cell_ranger")
        sc.pp.highly_variable_genes(
            adata_merged, layer="counts", n_top_genes=N_HVG,
            flavor="cell_ranger", subset=False
        )
        hvg_method = "cell_ranger"

print(f"  -> Method: {hvg_method}")

if FORCE_MARKERS_IN_HVG:
    n_added    = 0
    marker_set = set(FORCED_MARKERS)
    for idx, symbol_base in enumerate(adata_merged.var["symbol_base"]):
        if symbol_base in marker_set:
            real_name = adata_merged.var_names[idx]
            if not adata_merged.var.loc[real_name, "highly_variable"]:
                adata_merged.var.loc[real_name, "highly_variable"] = True
                n_added += 1
    print(f"  -> Forced {n_added}/{len(FORCED_MARKERS)} markers into HVG")

n_hvg_final = adata_merged.var["highly_variable"].sum()
print(f"  -> Final HVG count: {n_hvg_final}")

hvg_genes = adata_merged.var_names[adata_merged.var["highly_variable"]].tolist()
with open(output_dir / f"{OUTPUT_PREFIX}_hvg_genes.txt", "w") as f:
    f.write("\n".join(hvg_genes))



[Step 8] Selecting HVGs...
  -> batch-aware failed (b'There are other near singularities as well. 0.09), trying standard...
  -> Method: standard_seurat_v3
  -> Forced 15/72 markers into HVG
  -> Final HVG count: 4015


## Cell 12 — Step 9: Build Full Matrix for .raw

In [13]:
print("\n[Step 9] Building full matrix for .raw...")
full_counts = adata_merged.layers["counts"]
if issparse(full_counts) and not isinstance(full_counts, csr_matrix):
    full_counts = csr_matrix(full_counts)
raw_var = adata_merged.var.copy()
print(f"  Full matrix shape: {full_counts.shape}")



[Step 9] Building full matrix for .raw...
  Full matrix shape: (166015, 33559)


## Cell 13 — Step 10: Create Training Subset (HVG only)

In [14]:
print("\n[Step 10] Creating training subset (HVG only)...")

hvg_mask = adata_merged.var["highly_variable"].values
X_hvg    = adata_merged.layers["counts"][:, hvg_mask]

if issparse(X_hvg) and not isinstance(X_hvg, csr_matrix):
    X_hvg = csr_matrix(X_hvg)

adata_train = sc.AnnData(
    X=X_hvg.copy(),
    obs=adata_merged.obs.copy(),
    var=adata_merged.var.iloc[hvg_mask].copy()
)
adata_train.var_names        = adata_merged.var_names[hvg_mask]
adata_train.layers["counts"] = adata_train.X

print(f"  Training data: {adata_train.shape}")

adata_train.obs["scanvi_labels"] = adata_train.obs["cell_type_fine"].astype(str)
adata_train.obs["scanvi_labels"] = adata_train.obs["scanvi_labels"].astype("category")

if UNLABELED_CATEGORY not in adata_train.obs["scanvi_labels"].cat.categories:
    adata_train.obs["scanvi_labels"] = adata_train.obs["scanvi_labels"].cat.add_categories(
        [UNLABELED_CATEGORY]
    )

print("  -> Label distribution:")
print(adata_train.obs["scanvi_labels"].value_counts())

gc.collect()



[Step 10] Creating training subset (HVG only)...
  Training data: (166015, 4015)
  -> Label distribution:
scanvi_labels
Unknown    111272
0           22724
1           17585
2           10651
3            3763
4              20
Name: count, dtype: int64


20

## Cell 14 — Step 11: CellTypist (full genes)
> **[BUG-NEW-3 P0 FIX]** CellTypist promoted to first-class output branch

In [15]:
print("\n[Step 11] Running CellTypist on full gene matrix...")
try:
    # BUG-NEW-3 FIX: run_celltypist_on_full_genes now returns the predictions object.
    celltypist_predictions = run_celltypist_on_full_genes(adata_merged)

    # BUG-NEW-3 FIX: export CellTypist as first-class output branch.
    # Writes celltypist_label_direct, celltypist_label_direct_filt,
    # obsm["celltypist_proba"], uns["celltypist_label_order"], uns["celltypist_direct"].
    export_celltypist_direct_results(
        adata_merged,
        celltypist_predictions,
        label_key=CELLTYPIST_DIRECT_LABEL_KEY,
        filt_key=CELLTYPIST_DIRECT_FILT_KEY,
        prefer_majority=CELLTYPIST_MAJORITY_VOTE,
        conf_threshold=CELLTYPIST_CONF_THRESHOLD,
        save_proba=CELLTYPIST_SAVE_PROBA,
    )

    # Back-fill all CellTypist-derived columns to adata_train
    cols_to_backfill = [
        "celltypist_pred",
        "celltypist_confidence",
        CELLTYPIST_DIRECT_LABEL_KEY,
        CELLTYPIST_DIRECT_FILT_KEY,
    ]
    if "celltypist_majority" in adata_merged.obs.columns:
        cols_to_backfill.append("celltypist_majority")

    for col in cols_to_backfill:
        if col in adata_merged.obs.columns:
            adata_train.obs[col] = adata_merged.obs.loc[adata_train.obs_names, col].values

    print("  -> CellTypist back-filled to adata_train")

except Exception as e:
    print(f"  WARNING: CellTypist failed: {e}")
    import traceback
    traceback.print_exc()



[Step 11] Running CellTypist on full gene matrix...

[CellTypist] Starting annotation on FULL gene matrix...
  -> Loading model from local path: /home/h2048/data/source/reference/celltypist_models/Immune_All_Low.pkl
  -> 6064/6639 model genes matched via symbol_base (direct var_names: 6183)


🔬 Input data has 166015 cells and 6064 genes
🔗 Matching reference genes in the model
🧬 6064 features used for prediction
⚖️ Scaling input data
🖋️ Predicting labels
✅ Prediction done!
👀 Can not detect a neighborhood graph, will construct one before the over-clustering
⛓️ Over-clustering input data with resolution set to 25
🗳️ Majority voting the predictions
✅ Majority voting done!


  -> CellTypist predictions added
Mast cells                  42817
Classical monocytes         38679
Alveolar macrophages        23612
Macrophages                 16792
Monocytes                    5814
Regulatory T cells           5701
DC2                          5513
Intermediate macrophages     4328
pDC                          3837
Non-classical monocytes      3118
Name: count, dtype: int64
  -> Exporting CellTypist direct output branch...
    celltypist_label_direct: 18 unique types
    celltypist_label_direct_filt: 35460 cells below conf threshold 0.5 -> 'Unknown'
    celltypist_proba saved: shape (166015, 98)
  -> CellTypist back-filled to adata_train


## Cell 15 — Step 12: scVI Training

In [16]:
print("\n[Step 12] Training scVI...")

setup_kwargs = {
    "layer":                      "counts",
    "batch_key":                  BATCH_KEY,
    "continuous_covariate_keys":  ["pct_counts_mt", "stress_score", "S_score", "G2M_score"],
    "categorical_covariate_keys": [TISSUE_KEY]
}
scvi.model.SCVI.setup_anndata(adata_train, **setup_kwargs)

scvi_model = scvi.model.SCVI(
    adata_train,
    n_latent=SCVI_N_LATENT,
    n_layers=SCVI_N_LAYERS,
    n_hidden=SCVI_N_HIDDEN,
    dropout_rate=SCVI_DROPOUT
)

train_kwargs = {
    "max_epochs":              MAX_EPOCHS_SCVI,
    "batch_size":              BATCH_SIZE,
    "early_stopping":          True,
    "early_stopping_patience": 30,
    "plan_kwargs":             {"lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
}
if gpu_available:
    train_kwargs["accelerator"] = "gpu"
    train_kwargs["devices"]     = 1

scvi_model.train(**train_kwargs)
print("  -> scVI training complete")



[Step 12] Training scVI...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 400/400: 100%|██████████| 400/400 [2:24:06<00:00, 20.63s/it, v_num=1, train_loss_step=669, train_loss_epoch=673]  

`Trainer.fit` stopped: `max_epochs=400` reached.


Epoch 400/400: 100%|██████████| 400/400 [2:24:06<00:00, 21.62s/it, v_num=1, train_loss_step=669, train_loss_epoch=673]
  -> scVI training complete


## Cell 16 — Step 13: scANVI Training

In [17]:
print("\n[Step 13] Training scANVI...")

scanvi_model = scvi.model.SCANVI.from_scvi_model(
    scvi_model,
    adata=adata_train,
    labels_key="scanvi_labels",
    unlabeled_category=UNLABELED_CATEGORY
)

scanvi_train_kwargs = {
    "max_epochs":              MAX_EPOCHS_SCANVI,
    "batch_size":              BATCH_SIZE,
    "early_stopping":          True,
    "early_stopping_patience": 20,
    "plan_kwargs":             {"lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY},
}
if gpu_available:
    scanvi_train_kwargs["accelerator"] = "gpu"
    scanvi_train_kwargs["devices"]     = 1

scanvi_model.train(**scanvi_train_kwargs)
print("  -> scANVI training complete")



[Step 13] Training scANVI...
INFO     Training for 200 epochs.                                                                                  


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Epoch 134/200:  67%|██████▋   | 134/200 [1:34:33<46:34, 42.34s/it, v_num=1, train_loss_step=749, train_loss_epoch=657]  
Monitored metric elbo_validation did not improve in the last 20 records. Best score: 689.227. Signaling Trainer to stop.
  -> scANVI training complete


## Cell 17 — Step 14: Export Latent Representations & Predictions
> **[BUG-NEW-1 P0 FIX]** `label_order` extracted from DataFrame columns, not `scanvi_model.labels_`

In [18]:
print("\n[Step 14] Exporting latent representations and predictions...")

# --- scANVI latent representation (index-aligned) ---
latent_scanvi  = scanvi_model.get_latent_representation(adata_train)
latent_df      = pd.DataFrame(
    latent_scanvi,
    index=adata_train.obs_names,
    columns=[f"scANVI_{i}" for i in range(latent_scanvi.shape[1])]
)
latent_aligned = latent_df.reindex(adata_merged.obs_names)

if latent_aligned.isna().any().any():
    raise ValueError("CRITICAL: Missing scANVI latent representation after reindex!")

adata_merged.obsm["X_scANVI"] = latent_aligned.values

# --- Hard label predictions ---
pred_labels  = scanvi_model.predict(adata_train)
pred_df      = pd.Series(pred_labels, index=adata_train.obs_names)
pred_aligned = pred_df.reindex(adata_merged.obs_names)
adata_merged.obs["scanvi_pred"] = pred_aligned.values

# --- Soft probability predictions ---
# BUG-NEW-1 FIX: scanvi_model.labels_ returns per-cell predictions (N_cells),
# NOT class names. label_order must come from predict(soft=True) DataFrame columns.
proba_raw = scanvi_model.predict(adata_train, soft=True)

if isinstance(proba_raw, pd.DataFrame):
    label_order = list(proba_raw.columns)
    proba       = proba_raw.values.astype(np.float32)
else:
    proba = np.asarray(proba_raw, dtype=np.float32)
    try:
        label_order = list(
            scanvi_model.adata_manager
            .get_state_registry("labels")
            .categorical_mapping
        )
    except Exception:
        label_order = [f"label_{i}" for i in range(proba.shape[1])]

proba_df      = pd.DataFrame(proba, index=adata_train.obs_names, columns=label_order)
proba_aligned = proba_df.reindex(adata_merged.obs_names)
adata_merged.obsm["scanvi_proba"]      = proba_aligned.values
adata_merged.obs["scanvi_confidence"]  = proba_aligned.values.max(axis=1)
adata_merged.uns["scanvi_label_order"] = list(label_order)

print("  -> scANVI predictions (top 15):")
print(adata_merged.obs["scanvi_pred"].value_counts().head(15))

compute_novelty_scores(adata_merged, proba_aligned)



[Step 14] Exporting latent representations and predictions...
  -> scANVI predictions (top 15):
scanvi_pred
0    111042
1     54959
2        14
Name: count, dtype: int64
  -> Computing novelty scores (entropy-based)...
    High novelty query cells: 11524


## Cell 18 — Step 15: Attach .raw (full gene set)

In [19]:
print("\n[Step 15] Attaching .raw (full gene set)...")
from anndata import AnnData

adata_merged.raw = AnnData(X=full_counts, obs=adata_merged.obs.copy(), var=raw_var)
print(f"  [OK] .raw attached: {adata_merged.raw.n_vars} genes")



[Step 15] Attaching .raw (full gene set)...
  [OK] .raw attached: 33559 genes


## Cell 19 — Step 16: Multiple UMAPs (scVI + scANVI)

In [20]:
print("\n[Step 16] Computing Multiple UMAPs...")

# --- scVI latent (index-aligned) ---
print("  -> Getting scVI latent representation...")
latent_scvi     = scvi_model.get_latent_representation(adata_train)
latent_scvi_df  = pd.DataFrame(
    latent_scvi,
    index=adata_train.obs_names,
    columns=[f"scVI_{i}" for i in range(latent_scvi.shape[1])]
)
latent_scvi_aligned = latent_scvi_df.reindex(adata_merged.obs_names)
if latent_scvi_aligned.isna().any().any():
    raise ValueError("CRITICAL: Missing scVI latent representation after reindex!")
adata_merged.obsm["X_scVI"] = latent_scvi_aligned.values

# --- Query-only Leiden for novelty detection ---
run_query_only_leiden(adata_merged, resolution=QUERY_LEIDEN_RESOLUTION)

# --- 16a: UMAP on scVI latent ---
print("  -> Computing UMAP on scVI latent...")
sc.pp.neighbors(adata_merged, use_rep="X_scVI",   n_neighbors=30,
                random_state=RANDOM_SEED, key_added="neighbors_scVI")
sc.tl.umap(adata_merged, random_state=RANDOM_SEED, neighbors_key="neighbors_scVI")
adata_merged.obsm["X_umap_scVI"] = adata_merged.obsm["X_umap"].copy()
print("     Saved to X_umap_scVI")

# --- 16b: UMAP on scANVI latent (DEFAULT) ---
print("  -> Computing UMAP on scANVI latent (DEFAULT)...")
sc.pp.neighbors(adata_merged, use_rep="X_scANVI", n_neighbors=30,
                random_state=RANDOM_SEED, key_added="neighbors_scANVI")
sc.tl.umap(adata_merged, random_state=RANDOM_SEED, neighbors_key="neighbors_scANVI")
adata_merged.obsm["X_umap_scANVI"] = adata_merged.obsm["X_umap"].copy()
adata_merged.obsm["X_umap"]        = adata_merged.obsm["X_umap_scANVI"].copy()
print("     Saved to X_umap_scANVI and X_umap (default)")

# Save UMAP operator for future query projection
umap_op_scanvi = UMAP(n_neighbors=30, n_components=2, min_dist=0.5, spread=1.0,
                      metric="euclidean", random_state=RANDOM_SEED)
umap_op_scanvi.fit(adata_merged.obsm["X_scANVI"])
joblib.dump(umap_op_scanvi, output_dir / f"{OUTPUT_PREFIX}_umap_scanvi_operator.joblib")
print("  -> UMAP operator (scANVI) saved")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(adata_merged.obs["scanvi_confidence"], bins=50, edgecolor="black")
ax.set_xlabel("Prediction Confidence")
ax.set_ylabel("Cell Count")
ax.set_title("scANVI Confidence Distribution")
plt.tight_layout()
plt.savefig(output_dir / f"{OUTPUT_PREFIX}_confidence_histogram.png", dpi=150)
plt.close()



[Step 16] Computing Multiple UMAPs...
  -> Getting scVI latent representation...

[Novelty Detection] Running query-only Leiden (resolution=1.0)...
  -> Found 18 query-only clusters
leiden_query
N/A      54743
qry_0    13340
qry_1    12997
qry_2    12932
qry_3    12734
qry_4    11302
qry_5    10190
qry_6     7812
qry_7     6505
qry_8     4496
Name: count, dtype: int64
  -> Computing UMAP on scVI latent...
     Saved to X_umap_scVI
  -> Computing UMAP on scANVI latent (DEFAULT)...
     Saved to X_umap_scANVI and X_umap (default)
  -> UMAP operator (scANVI) saved


## Cell 19.5 — Step 16.5: Run Log & AnnData Structure Summary

In [21]:
print("\n[Step 16.5] Writing run log and AnnData structure summary...")

log_timestamp = datetime.now().isoformat()
log_path      = output_dir / f"{OUTPUT_PREFIX}_run_log.txt"
summary_path  = output_dir / f"{OUTPUT_PREFIX}_anndata_structure.json"
output_h5ad   = output_dir / f"{OUTPUT_PREFIX}_results.h5ad"


def _shape_list(obj):
    shape = getattr(obj, "shape", None)
    return [int(x) for x in shape] if shape is not None else None


def _mapping_summary(mapping):
    summary = {}
    for key in sorted(mapping.keys()):
        value = mapping[key]
        item  = {"shape": _shape_list(value)}
        dtype = getattr(value, "dtype", None)
        if dtype is not None:
            item["dtype"] = str(dtype)
        if hasattr(value, "shape"):
            item["is_sparse"] = bool(issparse(value))
        summary[str(key)] = item
    return summary


pipeline_log = {
    "timestamp":          log_timestamp,
    "reference_h5ad":     REFERENCE_H5AD,
    "query_h5ad":         QUERY_H5AD,
    "output_dir":         str(output_dir),
    "output_prefix":      OUTPUT_PREFIX,
    "planned_output_h5ad": str(output_h5ad),
    "n_obs":              int(adata_merged.n_obs),
    "n_vars":             int(adata_merged.n_vars),
    "n_hvg":              int(n_hvg_final),
    "gpu_available":      bool(gpu_available),
}
future_uns_keys = sorted(
    {str(k) for k in adata_merged.uns.keys()} |
    {"pipeline_log", "anndata_structure"}
)

anndata_structure = {
    "timestamp":            log_timestamp,
    "planned_output_h5ad":  str(output_h5ad),
    "shape":                {"n_obs": int(adata_merged.n_obs), "n_vars": int(adata_merged.n_vars)},
    "raw":                  {"present": adata_merged.raw is not None,
                             "shape": _shape_list(adata_merged.raw) if adata_merged.raw is not None else None},
    "obs_columns":          [str(c) for c in adata_merged.obs.columns],
    "var_columns":          [str(c) for c in adata_merged.var.columns],
    "layers":               _mapping_summary(adata_merged.layers),
    "obsm":                 _mapping_summary(adata_merged.obsm),
    "varm":                 _mapping_summary(adata_merged.varm),
    "obsp":                 _mapping_summary(adata_merged.obsp),
    "uns_keys":             future_uns_keys,
}

adata_merged.uns["pipeline_log"]       = pipeline_log
adata_merged.uns["anndata_structure"]  = anndata_structure

log_lines = [
    f"timestamp: {log_timestamp}",
    f"reference_h5ad: {REFERENCE_H5AD}",
    f"query_h5ad: {QUERY_H5AD}",
    f"output_dir: {output_dir}",
    f"output_prefix: {OUTPUT_PREFIX}",
    f"planned_output_h5ad: {output_h5ad}",
    f"shape: ({adata_merged.n_obs}, {adata_merged.n_vars})",
    f"raw_shape: {tuple(adata_merged.raw.shape) if adata_merged.raw is not None else 'None'}",
    f"layers: {', '.join(sorted(adata_merged.layers.keys())) or 'None'}",
    f"obsm: {', '.join(sorted(adata_merged.obsm.keys())) or 'None'}",
    f"obsp: {', '.join(sorted(adata_merged.obsp.keys())) or 'None'}",
    f"uns_keys: {', '.join(future_uns_keys) or 'None'}",
]

log_path.write_text("\n".join(log_lines) + "\n", encoding="utf-8")
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(anndata_structure, f, indent=2, ensure_ascii=False)

print(f"  -> Log written: {log_path}")
print(f"  -> Structure summary written: {summary_path}")
print("  -> Stored in adata_merged.uns['pipeline_log'] and adata_merged.uns['anndata_structure']")



[Step 16.5] Writing run log and AnnData structure summary...
  -> Log written: /home/h2048/data/py/20260308/myeloid_only_merged_pipeline/myeloid_only_merged_run_log.txt
  -> Structure summary written: /home/h2048/data/py/20260308/myeloid_only_merged_pipeline/myeloid_only_merged_anndata_structure.json
  -> Stored in adata_merged.uns['pipeline_log'] and adata_merged.uns['anndata_structure']


## Cell 20 — Step 17: Save Results

In [22]:
print("\n[Step 17] Saving Results...")

# P1 FIX (v1.4): convert label columns to category before write_h5ad.
# Reduces file size and speeds up groupby / plotting operations.
categorical_cols = [
    "data_source",
    "cell_type_coarse",
    "cell_type_fine",
    "scanvi_labels",
    "scanvi_pred",
    "mono_subtype_by_score",
    "leiden_query",
    CELLTYPIST_DIRECT_LABEL_KEY,
    CELLTYPIST_DIRECT_FILT_KEY,
]

for col in categorical_cols:
    if col in adata_merged.obs.columns:
        adata_merged.obs[col] = adata_merged.obs[col].astype("category")
    if col in adata_train.obs.columns:
        adata_train.obs[col] = adata_train.obs[col].astype("category")

scanvi_model.save(output_dir / f"{OUTPUT_PREFIX}_scanvi_model", overwrite=True)
scvi_model.save(output_dir / f"{OUTPUT_PREFIX}_scvi_model",    overwrite=True)

config = {
    "version":   "1.4",
    "timestamp": datetime.now().isoformat(),
    "input":     {"reference": REFERENCE_H5AD, "query": QUERY_H5AD},
    "n_hvg":     int(n_hvg_final),
    "scanvi_labels": list(label_order),
    "annotation_layers": {
        "celltypist_direct": {
            "label_key":            CELLTYPIST_DIRECT_LABEL_KEY,
            "filtered_label_key":   CELLTYPIST_DIRECT_FILT_KEY,
            "confidence_key":       "celltypist_confidence",
            "probability_key":      "celltypist_proba",
            "label_order_key":      "celltypist_label_order",
            "confidence_threshold": CELLTYPIST_CONF_THRESHOLD,
            "description":          "Pure CellTypist output on all merged cells",
        },
        "scanvi_reference": {
            "labels_key":     "scanvi_labels",
            "prediction_key": "scanvi_pred",
            "confidence_key": "scanvi_confidence",
            "latent_key":     "X_scANVI",
            "description":    "Reference-supervised scANVI; query cells = Unknown",
        },
    },
    "myeloid_specific": {
        "classical_mono_markers":    CLASSICAL_MONO_MARKERS,
        "nonclassical_mono_markers": NONCLASSICAL_MONO_MARKERS,
        "macrophage_markers":        MACROPHAGE_MARKERS,
        "dc_markers":                CDC1_MARKERS + CDC2_MARKERS,
        "neutrophil_markers":        NEUTROPHIL_MARKERS,
        "forced_markers_count":      len(FORCED_MARKERS),
    },
    "umap_spaces": {
        "X_umap":        "DEFAULT - scANVI-based UMAP",
        "X_umap_scVI":   "scVI latent space UMAP",
        "X_umap_scANVI": "scANVI latent space UMAP",
    },
    "bug_fixes": {
        "BUG-NEW-3_P0":      "CellTypist first-class output branch (v1.3)",
        "BUG-NEW-4_P1":      "BATCH_KEY prefixed ref_/qry_ before sc.concat (v1.3)",
        "P0_v1.4_proba_idx": "celltypist_proba explicitly index-aligned via reindex (v1.4)",
        "P1_v1.4_batch_pre": "BATCH_KEY existence checked before prefixing in Step 6 (v1.4)",
        "P1_v1.4_category":  "label columns categorized before write_h5ad (v1.4)",
        "BUG-NEW-1_P0":      "label_order from predict(soft=True) columns, not scanvi_model.labels_",
        "BUG-NEW-2_P1":      "os.environ thread vars set before import torch",
        "NEW-BUG-A_P0":      "Input paths corrected (were T cell paths in v1.0)",
        "NEW-BUG-B_P1":      "CD16 removed from NONCLASSICAL_MONO_MARKERS (not HGNC symbol)",
        "NEW-BUG-C_P2":      "compute_monocyte_subtype_scores vectorized with np.select",
        "BUG-1_P0":          "module score uses direct mean-expression (no score_genes)",
        "BUG-2_P0":          "CellTypist Model.load uses local-path check before download_models",
        "BUG-3_P1":          "obsm latent representations use pandas reindex",
        "BUG-4_P1":          "query_mask.values for numpy boolean indexing",
        "BUG-6_P0":          "sparse .X sampling uses .toarray().flatten()",
    },
}

with open(output_dir / f"{OUTPUT_PREFIX}_config.json", "w") as f:
    json.dump(config, f, indent=2)

output_h5ad = output_dir / f"{OUTPUT_PREFIX}_results.h5ad"
adata_merged.write_h5ad(output_h5ad, compression="gzip")
print(f"  -> Saved: {output_h5ad}")

train_h5ad = output_dir / f"{OUTPUT_PREFIX}_train_HVG.h5ad"
adata_train.write_h5ad(train_h5ad, compression="gzip")
print(f"  -> Saved: {train_h5ad}")



[Step 17] Saving Results...


... storing 'phase' as categorical
... storing 'celltypist_pred' as categorical
... storing 'celltypist_majority' as categorical
... storing 'symbol_base' as categorical
... storing 'symbol_base' as categorical


  -> Saved: /home/h2048/data/py/20260308/myeloid_only_merged_pipeline/myeloid_only_merged_results.h5ad


... storing 'phase' as categorical
... storing 'celltypist_pred' as categorical
... storing 'celltypist_majority' as categorical
... storing 'symbol_base' as categorical


  -> Saved: /home/h2048/data/py/20260308/myeloid_only_merged_pipeline/myeloid_only_merged_train_HVG.h5ad


## Cell 21 — Step 18: Myeloid-Specific Visualization

In [23]:
print("\n[Step 18] Creating Myeloid visualizations...")

# --- Overview figure (5 x 4 panels) ---
fig = plt.figure(figsize=(24, 20))
gs  = fig.add_gridspec(5, 4, hspace=0.3, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
sc.pl.umap(adata_merged, color="data_source", ax=ax1, show=False,
           title="Data Source (scANVI)", s=15)

ax2 = fig.add_subplot(gs[0, 1])
adata_merged.obs["_ref_coarse"] = pd.Series(pd.NA, index=adata_merged.obs_names, dtype="object")
mask_ref = adata_merged.obs["data_source"] == "reference"
adata_merged.obs.loc[mask_ref, "_ref_coarse"] = (
    adata_merged.obs.loc[mask_ref, "cell_type_coarse"].astype(str).values
)
adata_merged.obs["_ref_coarse"] = adata_merged.obs["_ref_coarse"].astype("category")
sc.pl.umap(adata_merged, color="_ref_coarse", ax=ax2, show=False,
           title="Reference Coarse Labels", legend_loc="on data", s=15)

ax3 = fig.add_subplot(gs[0, 2])
sc.pl.umap(adata_merged, color="scanvi_pred", ax=ax3, show=False,
           title="scANVI Predictions", legend_loc="on data", s=15)

ax4 = fig.add_subplot(gs[0, 3])
sc.pl.umap(adata_merged, color="scanvi_confidence", ax=ax4, show=False,
           title="Confidence", cmap="viridis", vmin=0, vmax=1, s=15)

ax5 = fig.add_subplot(gs[1, 0])
sc.pl.umap(adata_merged, color="Classical_Mono_score", ax=ax5, show=False,
           title="Classical Mono Score (CD14)", cmap="Reds", s=15)

ax6 = fig.add_subplot(gs[1, 1])
sc.pl.umap(adata_merged, color="NonClassical_Mono_score", ax=ax6, show=False,
           title="NonClassical Mono Score (FCGR3A)", cmap="Blues", s=15)

ax7 = fig.add_subplot(gs[1, 2])
sc.pl.umap(adata_merged, color="mono_subtype_by_score", ax=ax7, show=False,
           title="Monocyte Subtype by Score", legend_loc="on data", s=15)

ax8 = fig.add_subplot(gs[1, 3])
query_mask_viz = adata_merged.obs["data_source"] == "query"
ax8.scatter(
    adata_merged.obs.loc[query_mask_viz, "Classical_Mono_score"],
    adata_merged.obs.loc[query_mask_viz, "NonClassical_Mono_score"],
    c=adata_merged.obs.loc[query_mask_viz, "scanvi_confidence"],
    cmap="viridis", s=5, alpha=0.5
)
ax8.axhline(y=NONCLASSICAL_MONO_THRESHOLD, color='k', linestyle='--', alpha=0.3)
ax8.axvline(x=CLASSICAL_MONO_THRESHOLD,    color='k', linestyle='--', alpha=0.3)
ax8.set_xlabel("Classical Mono Score")
ax8.set_ylabel("NonClassical Mono Score")
ax8.set_title(f"Query: Mono Subtype Scores (thr={CLASSICAL_MONO_THRESHOLD})")

marker_panels = [
    ("LYZ",    gs[2, 0], "LYZ (pan-Myeloid)"),
    ("CD14",   gs[2, 1], "CD14 (Classical Mono)"),
    ("FCGR3A", gs[2, 2], "FCGR3A/CD16 (NonClassical Mono)"),
    ("CD68",   gs[2, 3], "CD68 (Macrophage)"),
    ("CD1C",   gs[3, 0], "CD1C (cDC2)"),
    ("CLEC9A", gs[3, 1], "CLEC9A (cDC1)"),
    ("S100A8", gs[3, 2], "S100A8 (Neutrophil)"),
    ("MKI67",  gs[3, 3], "MKI67 (Proliferating)"),
]
for gene, gs_pos, title in marker_panels:
    ax = fig.add_subplot(gs_pos)
    if gene in adata_merged.raw.var_names:
        sc.pl.umap(adata_merged, color=gene, ax=ax, show=False,
                   title=title, cmap="Reds", s=15, use_raw=True)

# --- CellTypist direct label panel (NEW in v1.3) ---
ax_ct1 = fig.add_subplot(gs[4, 0])
if CELLTYPIST_DIRECT_LABEL_KEY in adata_merged.obs.columns:
    sc.pl.umap(adata_merged, color=CELLTYPIST_DIRECT_LABEL_KEY, ax=ax_ct1, show=False,
               title="CellTypist Direct Labels", legend_loc="on data", s=15)
else:
    ax_ct1.set_title("CellTypist Direct (not available)")

ax_ct2 = fig.add_subplot(gs[4, 1])
if CELLTYPIST_DIRECT_FILT_KEY in adata_merged.obs.columns:
    sc.pl.umap(adata_merged, color=CELLTYPIST_DIRECT_FILT_KEY, ax=ax_ct2, show=False,
               title=f"CellTypist Direct (conf>={CELLTYPIST_CONF_THRESHOLD})",
               legend_loc="on data", s=15)
else:
    ax_ct2.set_title("CellTypist Filtered (not available)")

ax17 = fig.add_subplot(gs[4, 2])
_scanvi_cats = sorted(adata_merged.obs["scanvi_pred"].dropna().unique())
ref_scanvi = adata_merged.obs.loc[
    adata_merged.obs["data_source"] == "reference", "scanvi_pred"
].value_counts().reindex(_scanvi_cats, fill_value=0)
qry_scanvi = adata_merged.obs.loc[
    adata_merged.obs["data_source"] == "query", "scanvi_pred"
].value_counts().reindex(_scanvi_cats, fill_value=0)
x     = np.arange(len(_scanvi_cats))
width = 0.35
ax17.bar(x - width/2, ref_scanvi.values, width, label="Reference", alpha=0.8)
ax17.bar(x + width/2, qry_scanvi.values, width, label="Query",     alpha=0.8)
ax17.set_xticks(x)
ax17.set_xticklabels(_scanvi_cats, rotation=45, ha="right")
ax17.set_ylabel("Cell Count")
ax17.set_title("Myeloid scANVI Prediction Distribution")
ax17.legend()

ax18 = fig.add_subplot(gs[4, 3])
sc.pl.umap(adata_merged, color="novelty_score", ax=ax18, show=False,
           title="Novelty Score (Query)", cmap="hot", vmin=0, vmax=1, s=15)

plt.savefig(output_dir / f"{OUTPUT_PREFIX}_myeloid_overview.pdf",
            dpi=300, bbox_inches="tight")
plt.close()
print(f"  -> Saved: {OUTPUT_PREFIX}_myeloid_overview.pdf")

# --- scVI vs scANVI UMAP comparison ---
fig2, axes2 = plt.subplots(2, 3, figsize=(18, 12))
for row, (basis_key, label) in enumerate([("X_umap_scVI", "scVI"), ("X_umap_scANVI", "scANVI")]):
    sc.pl.embedding(
        adata_merged, basis=basis_key, color="data_source",
        ax=axes2[row, 0], show=False,
        title=f"Data Source ({label} UMAP)", s=10
    )
    sc.pl.embedding(
        adata_merged, basis=basis_key, color="_ref_coarse",
        ax=axes2[row, 1], show=False,
        title=f"Reference Labels ({label})", legend_loc="on data", s=10
    )
    sc.pl.embedding(
        adata_merged, basis=basis_key, color="scanvi_pred",
        ax=axes2[row, 2], show=False,
        title=f"scANVI Predictions ({label})", legend_loc="on data", s=10
    )
plt.tight_layout()
plt.savefig(output_dir / f"{OUTPUT_PREFIX}_umap_comparison.pdf",
            dpi=300, bbox_inches="tight")
plt.close()
print(f"  -> Saved: {OUTPUT_PREFIX}_umap_comparison.pdf")

# --- Novelty analysis ---
fig3, axes3 = plt.subplots(2, 2, figsize=(14, 12))
qry_mask = adata_merged.obs["data_source"] == "query"

ax = axes3[0, 0]
ax.hist(adata_merged.obs.loc[qry_mask, "novelty_score"],
        bins=50, edgecolor="black", alpha=0.7)
ax.axvline(x=0.7, color='r', linestyle='--', label='High novelty threshold')
ax.set_xlabel("Novelty Score")
ax.set_ylabel("Cell Count")
ax.set_title("Query Cells: Novelty Score Distribution")
ax.legend()

ax = axes3[0, 1]
scatter = ax.scatter(
    adata_merged.obs.loc[qry_mask, "scanvi_confidence"],
    adata_merged.obs.loc[qry_mask, "novelty_score"],
    c=adata_merged.obs.loc[qry_mask, "scanvi_entropy"],
    cmap="viridis", s=5, alpha=0.5
)
ax.set_xlabel("scANVI Confidence")
ax.set_ylabel("Novelty Score")
ax.set_title("Query: Confidence vs Novelty (colored by entropy)")
plt.colorbar(scatter, ax=ax)

ax = axes3[1, 0]
leiden_counts = adata_merged.obs.loc[qry_mask, "leiden_query"].value_counts().head(15)
ax.barh(range(len(leiden_counts)), leiden_counts.values)
ax.set_yticks(range(len(leiden_counts)))
ax.set_yticklabels(leiden_counts.index)
ax.set_xlabel("Cell Count")
ax.set_title("Query-only Leiden Clusters (Top 15)")

ax = axes3[1, 1]
ax.scatter(
    adata_merged.obs.loc[qry_mask, "Macrophage_score"],
    adata_merged.obs.loc[qry_mask, "Neutrophil_score"],
    c=adata_merged.obs.loc[qry_mask, "scanvi_confidence"],
    cmap="viridis", s=5, alpha=0.5
)
ax.set_xlabel("Macrophage Score")
ax.set_ylabel("Neutrophil Score")
ax.set_title("Query: Macrophage vs Neutrophil Score")

plt.tight_layout()
plt.savefig(output_dir / f"{OUTPUT_PREFIX}_novelty_analysis.pdf",
            dpi=300, bbox_inches="tight")
plt.close()
print(f"  -> Saved: {OUTPUT_PREFIX}_novelty_analysis.pdf")

adata_merged.obs.drop(columns=["_ref_coarse"], inplace=True, errors="ignore")

# --- Pipeline summary ---
print("\n" + "=" * 80)
print("MYELOID PIPELINE COMPLETE (v1.3)")
print("=" * 80)
print(f"Output: {output_dir}")
print(f"  Total cells : {adata_merged.n_obs:,}")
print(f"  Reference   : {(adata_merged.obs['data_source'] == 'reference').sum():,}")
print(f"  Query       : {(adata_merged.obs['data_source'] == 'query').sum():,}")

print("\nMonocyte Subtype Distribution in Query (by score):")
print(adata_merged.obs.loc[qry_mask, "mono_subtype_by_score"].value_counts())

print("\nTop scANVI Predictions in Query:")
print(adata_merged.obs.loc[qry_mask, "scanvi_pred"].value_counts().head(10))

if CELLTYPIST_DIRECT_LABEL_KEY in adata_merged.obs.columns:
    print(f"\nTop CellTypist Direct Labels (all cells):")
    print(adata_merged.obs[CELLTYPIST_DIRECT_LABEL_KEY].value_counts().head(10))

print(f"\nHigh novelty cells (score > 0.7) : {adata_merged.obs['is_potentially_novel'].sum():,}")
print(f"Query-only Leiden clusters         : {adata_merged.obs.loc[qry_mask, 'leiden_query'].nunique()}")

print("\nBug Fixes Applied in v1.3:")
print("  BUG-NEW-3 [P0] CellTypist first-class output: celltypist_label_direct, _filt, proba")
print("  BUG-NEW-4 [P1] BATCH_KEY prefixed ref_/qry_ before sc.concat")
print("  (inherited v1.2): BUG-NEW-1/2, NEW-BUG-A/B/C")
print("=" * 80)



[Step 18] Creating Myeloid visualizations...
  -> Saved: myeloid_only_merged_myeloid_overview.pdf
  -> Saved: myeloid_only_merged_umap_comparison.pdf
  -> Saved: myeloid_only_merged_novelty_analysis.pdf

MYELOID PIPELINE COMPLETE (v1.3)
Output: /home/h2048/data/py/20260308/myeloid_only_merged_pipeline
  Total cells : 166,015
  Reference   : 54,743
  Query       : 111,272

Monocyte Subtype Distribution in Query (by score):
mono_subtype_by_score
Negative          60837
NonClassical      28236
Classical         10493
Intermediate       9474
DoublePositive     2232
Name: count, dtype: int64

Top scANVI Predictions in Query:
scanvi_pred
0    80536
1    30724
2       12
Name: count, dtype: int64

Top CellTypist Direct Labels (all cells):
celltypist_label_direct
Classical monocytes         53793
Mast cells                  43538
Alveolar macrophages        25245
Macrophages                 19443
DC2                          7001
Intermediate macrophages     4356
pDC                          